In [1]:
%pip install --upgrade chromadb openai numpy python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#importing libraries
import os
import chromadb

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client=OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
#Preparing our documents

documents = [
    "Python is a popular programming language.",
    "Machine learning allows computers to learn from data.",
    "RAG combines document retrieval with language generation.",
    "Neural networks are inspired by the human brain.",
    "Embeddings represent text as numerical vectors.",
    "Vector databases store embeddings for similarity search."
]

In [5]:
#Creating IDs and Metadata
ids = [
    "doc1",
    "doc2",
    "doc3",
    "doc4",
    "doc5",
    "doc6"
]

metadatas = [
    {"topic": "Python", "source": "learning_notes"},
    {"topic": "Machine Learning", "source": "learning_notes"},
    {"topic": "RAG", "source": "learning_notes"},
    {"topic": "Neural Networks", "source": "learning_notes"},
    {"topic": "Embeddings", "source": "learning_notes"},
    {"topic": "Vector Databases", "source": "learning_notes"}
]

In [6]:
#Generating Embeddings

def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

In [7]:
embeddings = []

for document in documents:

    embedding = get_embedding(document)

    embeddings.append(embedding)

print("Number of embeddings:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

Number of embeddings: 6
Vector dimensions: 1536


In [8]:
#Creating Chromadb database

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [9]:
#Creating a collection

collection = chroma_client.get_or_create_collection(
    name="ai_learning_collection"
)

In [10]:
#Storing documents in ChromaDB
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

In [11]:
print("Total records:", collection.count())

Total records: 6


In [12]:
#Giving query and searching it in ChromaDB collection
query = "How can AI search documents using meaning?"

query_embedding = get_embedding(query)

In [13]:
#Sending the query embedding to ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [14]:
for i in range(len(results["ids"][0])):

    print("Document:", results["documents"][0][i])

    print("Metadata:", results["metadatas"][0][i])

    print("Distance:", results["distances"][0][i])

    print("------------------------")

Document: RAG combines document retrieval with language generation.
Metadata: {'topic': 'RAG', 'source': 'learning_notes'}
Distance: 0.9987974166870117
------------------------
Document: Vector databases store embeddings for similarity search.
Metadata: {'source': 'learning_notes', 'topic': 'Vector Databases'}
Distance: 1.2995001077651978
------------------------
Document: Machine learning allows computers to learn from data.
Metadata: {'source': 'learning_notes', 'topic': 'Machine Learning'}
Distance: 1.353348970413208
------------------------


In [16]:
#Filter results using metadata
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"topic": "RAG"}
)

results

{'ids': [['doc3']],
 'embeddings': None,
 'documents': [['RAG combines document retrieval with language generation.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'learning_notes', 'topic': 'RAG'}]],
 'distances': [[0.9987974166870117]]}